# NinaPro EMG Explorer

Load a specific subject from a NinaPro dataset and inspect the data.

In [ ]:
## Configuration
DB       = 2          # dataset number (1–5)
SUBJECTS = [1, 2, 3]  # subjects to evaluate

# --- Hyperparameter search space ---
# CONV_CONFIG: tuples of (k1, k2, k3) kernel sizes for the three parallel branches
CONV_CONFIG           = [(3, 5, 11), (5, 9, 19), (7, 9, 13), (3, 11, 23)]
WINDOW_SIZES          = [256, 512]   # samples  (128 ms / 256 ms @ 2 kHz)
NUM_MULTISCALE_STAGES = [1, 2]       # how many multi-scale blocks stacked in sequence

MLFLOW_EXPERIMENT = "ninapro_emg"    # MLflow experiment name

In [ ]:
import glob
from pathlib import Path

import numpy as np
import scipy.io as sio

NINAPRO_DIR = Path("ninapro")

def load_subject(db: int, subject: int, ninapro_dir: Path = NINAPRO_DIR) -> dict[str, np.ndarray]:
    """Load all exercise .mat files for a subject and concatenate them.

    Returns a dict with keys: emg, stimulus, restimulus, repetition.
    All arrays are (n_samples, n_channels) or (n_samples,) for labels.
    """
    subject_dir = ninapro_dir / f"DB{db}" / f"DB{db}_s{subject}"
    patterns = [
        str(subject_dir / f"S{subject}_E*_A*.mat"),
        str(subject_dir / f"s{subject}_E*_A*.mat"),
        str(subject_dir / f"DB{db}_s{subject}_E*_A*.mat"),
        str(subject_dir / f"DB{db}_S{subject}_E*_A*.mat"),
    ]
    mat_files = []
    for p in patterns:
        mat_files.extend(sorted(glob.glob(p)))

    if not mat_files:
        raise FileNotFoundError(
            f"No .mat files found for DB{db} subject {subject} in {subject_dir}.\n"
            f"Run: python get_ninapro.py --db {db} --subjects {subject}\n"
            f"Current working directory: {Path.cwd()}"
        )

    arrays: dict[str, list] = {"emg": [], "stimulus": [], "restimulus": [], "repetition": []}
    for path in mat_files:
        mat = sio.loadmat(path, squeeze_me=True)
        for key in arrays:
            if key in mat:
                arrays[key].append(np.atleast_1d(mat[key]))

    return {k: np.concatenate(v) if v else np.array([]) for k, v in arrays.items()}


# Preview the first subject
data = load_subject(DB, SUBJECTS[0])
print(f"Loaded DB{DB} subject {SUBJECTS[0]}")
for key, arr in data.items():
    print(f"  {key:12s}: shape={arr.shape}, dtype={arr.dtype}")

In [ ]:
import matplotlib.pyplot as plt

# Signal preview for the first subject
emg = data["emg"]
stimulus = data["stimulus"]
n_channels = emg.shape[1] if emg.ndim == 2 else 1
t = np.arange(len(emg))

fig, axes = plt.subplots(n_channels + 1, 1, figsize=(14, 2 * (n_channels + 1)), sharex=True)

for ch in range(n_channels):
    axes[ch].plot(t, emg[:, ch] if emg.ndim == 2 else emg, lw=0.4)
    axes[ch].set_ylabel(f"Ch {ch+1}")

axes[-1].plot(t, stimulus, color="tab:red", lw=0.8)
axes[-1].set_ylabel("Stimulus")
axes[-1].set_xlabel("Sample")

fig.suptitle(f"DB{DB} — Subject {SUBJECTS[0]}", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter

# Class overview for the first subject
labels_preview = data["restimulus"] if data["restimulus"].size else data["stimulus"]
classes = np.unique(labels_preview[labels_preview > 0])
print(f"Movement classes in subject {SUBJECTS[0]}: {classes}")

## Labeled window extraction

Slide a fixed-length window over the EMG signal and assign the majority label within each window.
`restimulus` (re-labeled after the experiment) is used instead of `stimulus` for cleaner boundaries.

In [ ]:
from collections import Counter

WINDOW_SIZE = 200   # samples  (e.g. 200 @ 2 kHz = 100 ms)
WINDOW_STEP = 100   # overlap stride

def extract_windows(
    emg: np.ndarray,
    labels: np.ndarray,
    window_size: int = WINDOW_SIZE,
    step: int = WINDOW_STEP,
    drop_rest: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    """Return (windows, labels) arrays.

    windows : (N, window_size, n_channels)
    labels  : (N,)  majority label within each window; windows where the
               majority label is 0 (rest) are dropped when drop_rest=True.
    """
    n_samples, n_channels = emg.shape
    starts = range(0, n_samples - window_size + 1, step)

    X, y = [], []
    for s in starts:
        window_labels = labels[s : s + window_size]
        majority = Counter(window_labels.tolist()).most_common(1)[0][0]
        if drop_rest and majority == 0:
            continue
        X.append(emg[s : s + window_size])
        y.append(majority)

    return np.stack(X).astype(np.float32), np.array(y, dtype=np.int64)


labels = data["restimulus"] if data["restimulus"].size else data["stimulus"]
X, y = extract_windows(data["emg"], labels)

print(f"Windows : {X.shape}   (N, window_size, channels)")
print(f"Labels  : {y.shape}")
print(f"Classes : {np.unique(y)}")
print(f"Distribution: { {int(k): int(v) for k, v in Counter(y.tolist()).most_common()} }")

## PyTorch Dataset & DataLoader

`NinaProDataset` loads one subject lazily (the `.mat` files are read once on first use) and serves windows on-the-fly via `__getitem__` — no giant pre-allocated tensor needed.

Set `subjects` to a list to combine multiple subjects into one dataset.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset


class NinaProDataset(Dataset):
    """Lazy-loading windowed EMG dataset for a single NinaPro subject.

    The subject's .mat files are loaded once on first __getitem__ call.
    Windows are sliced on-the-fly, so only one subject's raw signal
    sits in memory at a time.

    Args:
        db:          Dataset number (1–5).
        subject:     Subject number.
        window_size: Window length in samples.
        step:        Stride between consecutive windows.
        drop_rest:   If True, windows whose majority label is 0 are excluded.
        ninapro_dir: Root directory containing DB{N}/ subdirectories.
        label_map:   Optional dict mapping original label ints to new ints.
                     Useful for remapping class indices to 0-based contiguous.
    """

    def __init__(
        self,
        db: int,
        subject: int,
        window_size: int = 200,
        step: int = 100,
        drop_rest: bool = True,
        ninapro_dir: Path = NINAPRO_DIR,
        label_map = None,
    ):
        self.db = db
        self.subject = subject
        self.window_size = window_size
        self.step = step
        self.drop_rest = drop_rest
        self.ninapro_dir = ninapro_dir
        self.label_map = label_map

        self._emg: np.ndarray | None = None
        self._labels: np.ndarray | None = None
        self._indices: list[int] | None = None  # valid window start positions

    def _load(self) -> None:
        data = load_subject(self.db, self.subject, self.ninapro_dir)
        self._emg = data["emg"].astype(np.float32)
        raw_labels = data["restimulus"] if data["restimulus"].size else data["stimulus"]

        valid_starts = []
        for s in range(0, len(self._emg) - self.window_size + 1, self.step):
            window_labels = raw_labels[s : s + self.window_size]
            majority = Counter(window_labels.tolist()).most_common(1)[0][0]
            if self.drop_rest and majority == 0:
                continue
            valid_starts.append(s)

        self._labels = raw_labels
        self._indices = valid_starts

    @property
    def emg(self) -> np.ndarray:
        if self._emg is None:
            self._load()
        return self._emg

    def __len__(self) -> int:
        if self._indices is None:
            self._load()
        return len(self._indices)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        if self._indices is None:
            self._load()
        start = self._indices[idx]
        window = self._emg[start : start + self.window_size]          # (W, C)
        majority = Counter(
            self._labels[start : start + self.window_size].tolist()
        ).most_common(1)[0][0]
        label = self.label_map[majority] if self.label_map else majority
        return torch.from_numpy(window), torch.tensor(label, dtype=torch.long)


# --- build label map so class indices are 0-based contiguous ---
# (NinaPro labels start at 1 for movements; 0 = rest, which is dropped)
all_labels = sorted(int(c) for c in np.unique(y))   # from the extracted windows above
label_map = {orig: new for new, orig in enumerate(all_labels)}
print("Label map (original → 0-based):", label_map)

dataset = NinaProDataset(
    db=DB,
    subject=SUBJECT,
    window_size=WINDOW_SIZE,
    step=WINDOW_STEP,
    label_map=label_map,
)
print(f"Dataset size: {len(dataset)} windows")

# Example: combine multiple subjects
# dataset = ConcatDataset([
#     NinaProDataset(DB, s, label_map=label_map) for s in [1, 2, 3]
# ])

loader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=0)

batch_x, batch_y = next(iter(loader))
print(f"Batch EMG  : {batch_x.shape}   (batch, window_size, channels)")
print(f"Batch labels: {batch_y.shape},  unique={batch_y.unique().tolist()}")

## Preprocessing & Classification Pipeline

Literature-grounded choices:
- **Bandpass 10–500 Hz** (4th-order Butterworth): removes DC drift and high-freq noise [Atzori et al. 2016, Côté-Allard et al. 2019]
- **Per-channel z-score normalisation** computed on the training set only, applied to both splits
- **Train/test split by repetition**: reps 1–4 → train, reps 5–6 → test. This is the standard protocol to avoid leakage from overlapping windows [Atzori et al. 2016]
- **1D multi-scale CNN**: parallel branches with kernel sizes 3 / 7 / 15 capture short and long temporal patterns; features are concatenated then refined — similar structure to Hu et al. (2018) and the multi-scale attention networks (2023)
- **BatchNorm + Dropout 0.5** after each block [standard across literature]
- **Adam + cosine annealing LR** [Côté-Allard et al. 2019, EMGBench 2024]

In [ ]:
import itertools
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from scipy.signal import butter, sosfiltfilt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score
import mlflow
import mlflow.pytorch

FS           = 2000   # Hz — NinaPro DB2–5; set to 100 for DB1
TRAIN_REPS   = [1, 2, 3, 4]
TEST_REPS    = [5, 6]
WINDOW_STEP  = 40     # 20 ms stride (fixed)
EPOCHS       = 50
LR           = 1e-3
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 64
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else
                            "mps"  if torch.backends.mps.is_available() else "cpu")
print(f"Device: {DEVICE}")

mlflow.set_tracking_uri("mlruns")   # local file-based store, no DB required
mlflow.set_experiment(MLFLOW_EXPERIMENT)


# ---------------------------------------------------------------------------
# Preprocessing helpers
# ---------------------------------------------------------------------------

def bandpass_filter(emg: np.ndarray, fs: int = FS, low: float = 10.0, high: float = 500.0) -> np.ndarray:
    sos = butter(4, [low, high], btype="bandpass", fs=fs, output="sos")
    return sosfiltfilt(sos, emg, axis=0).astype(np.float32)


class NinaProDatasetByRepetition(Dataset):
    def __init__(self, db, subject, reps, window_size, step=WINDOW_STEP,
                 drop_rest=True, ninapro_dir=NINAPRO_DIR, label_map=None,
                 channel_stats=None, fs=FS):
        self.window_size = window_size
        self.label_map = label_map

        raw = load_subject(db, subject, ninapro_dir)
        emg = bandpass_filter(raw["emg"].astype(np.float32), fs=fs)
        labels     = raw["restimulus"] if raw["restimulus"].size else raw["stimulus"]
        repetition = raw["repetition"].astype(int)

        mask = np.isin(repetition, reps)
        emg, labels = emg[mask], labels[mask]

        if channel_stats is None:
            self.mean = emg.mean(axis=0, keepdims=True)
            self.std  = emg.std(axis=0, keepdims=True).clip(min=1e-8)
        else:
            self.mean, self.std = channel_stats
        emg = (emg - self.mean) / self.std

        self._emg = emg
        self._windows = []
        for s in range(0, len(emg) - window_size + 1, step):
            majority = Counter(labels[s : s + window_size].tolist()).most_common(1)[0][0]
            if drop_rest and majority == 0:
                continue
            self._windows.append((s, majority))

    @property
    def channel_stats(self):
        return self.mean, self.std

    def __len__(self):
        return len(self._windows)

    def __getitem__(self, idx):
        start, majority = self._windows[idx]
        window = self._emg[start : start + self.window_size]
        label = self.label_map[majority] if self.label_map else majority
        return torch.from_numpy(window.T), torch.tensor(label, dtype=torch.long)


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------

class ConvBnRelu(nn.Sequential):
    def __init__(self, in_ch, out_ch, kernel):
        super().__init__(
            nn.Conv1d(in_ch, out_ch, kernel, padding=kernel // 2, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(inplace=True),
        )


class MultiScaleBlock(nn.Module):
    """One multi-scale block: three parallel conv branches → concat → 1×1 projection.

    in_ch  → out_ch channels (projection ensures stable channel count across stacked blocks).
    """
    def __init__(self, in_ch: int, out_ch: int, kernels: tuple[int, int, int], dropout: float = 0.0):
        super().__init__()
        branch_ch = out_ch // 3  # each branch gets out_ch/3 filters
        remainder = out_ch - branch_ch * 3  # handle non-divisible by 3

        self.branches = nn.ModuleList([
            ConvBnRelu(in_ch, branch_ch + (1 if i < remainder else 0), k)
            for i, k in enumerate(kernels)
        ])
        # 1×1 conv to fuse and normalise channel count
        self.proj = nn.Sequential(
            nn.Conv1d(out_ch, out_ch, 1, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(torch.cat([b(x) for b in self.branches], dim=1))


class MultiScaleEMGNet(nn.Module):
    """Stacked multi-scale CNN for EMG gesture classification.

    Args:
        n_channels:   Number of EMG electrode channels (input channels).
        num_classes:  Number of gesture classes.
        kernels:      Tuple of 3 kernel sizes for the parallel branches.
        num_stages:   How many MultiScaleBlocks to stack sequentially.
        dropout:      Dropout probability inside each block.
    """

    def __init__(
        self,
        n_channels: int,
        num_classes: int,
        kernels: tuple[int, int, int] = (3, 7, 15),
        num_stages: int = 1,
        dropout: float = 0.5,
    ):
        super().__init__()
        base = 192  # divisible by 3 → equal branch widths

        stages = []
        in_ch = n_channels
        for i in range(num_stages):
            stages.append(MultiScaleBlock(in_ch, base, kernels,
                                          dropout=dropout if i == num_stages - 1 else 0.0))
            if i < num_stages - 1:
                stages.append(nn.MaxPool1d(2))  # downsample between stages
            in_ch = base
        self.backbone = nn.Sequential(*stages)

        self.refine = nn.Sequential(
            ConvBnRelu(base, 256, 3),
            nn.MaxPool1d(2),
            ConvBnRelu(256, 256, 3),
            nn.Dropout(dropout),
        )
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(256, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone(x)
        x = self.refine(x)
        return self.classifier(self.gap(x).squeeze(-1))


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------

def train_and_evaluate(
    db: int,
    subject: int,
    window_size: int,
    kernels: tuple[int, int, int],
    num_stages: int,
) -> dict:
    """Train and evaluate one subject with given hyperparameters.
    Logs everything to MLflow. Returns a result dict.
    """
    run_name = f"db{db}_s{subject}_w{window_size}_k{'_'.join(map(str,kernels))}_stages{num_stages}"

    with mlflow.start_run(run_name=run_name):
        # --- log hyperparameters ---
        mlflow.log_params({
            "db":          db,
            "subject":     subject,
            "window_size": window_size,
            "kernels":     str(kernels),
            "num_stages":  num_stages,
            "epochs":      EPOCHS,
            "lr":          LR,
            "weight_decay": WEIGHT_DECAY,
            "batch_size":  BATCH_SIZE,
            "train_reps":  str(TRAIN_REPS),
            "test_reps":   str(TEST_REPS),
        })

        # --- label map from training reps only ---
        tmp        = load_subject(db, subject)
        tmp_labels = tmp["restimulus"] if tmp["restimulus"].size else tmp["stimulus"]
        tmp_reps   = tmp["repetition"].astype(int)
        train_mask = np.isin(tmp_reps, TRAIN_REPS)
        all_labels = sorted(int(c) for c in np.unique(tmp_labels[train_mask]) if c > 0)
        label_map  = {orig: new for new, orig in enumerate(all_labels)}
        num_classes = len(label_map)

        train_ds = NinaProDatasetByRepetition(
            db, subject, TRAIN_REPS, window_size=window_size, label_map=label_map)
        test_ds  = NinaProDatasetByRepetition(
            db, subject, TEST_REPS,  window_size=window_size, label_map=label_map,
            channel_stats=train_ds.channel_stats)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
        test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

        mlflow.log_params({"num_classes": num_classes,
                           "train_windows": len(train_ds),
                           "test_windows":  len(test_ds)})

        n_channels = train_ds[0][0].shape[0]
        model = MultiScaleEMGNet(n_channels, num_classes, kernels=kernels, num_stages=num_stages).to(DEVICE)
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        mlflow.log_param("n_params", n_params)

        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

        history = {"train_acc": [], "test_acc": []}

        for epoch in range(1, EPOCHS + 1):
            for phase, loader, train in [("train", train_loader, True), ("test", test_loader, False)]:
                model.train(train)
                correct = n = 0
                with torch.set_grad_enabled(train):
                    for x, y in loader:
                        x, y = x.to(DEVICE), y.to(DEVICE)
                        logits = model(x)
                        loss = criterion(logits, y)
                        if train:
                            optimizer.zero_grad(); loss.backward(); optimizer.step()
                        correct += (logits.argmax(1) == y).sum().item()
                        n += len(y)
                acc = correct / n
                history[f"{phase}_acc"].append(acc)
                mlflow.log_metric(f"{phase}_acc", acc, step=epoch)
            scheduler.step()

        # --- final predictions ---
        model.eval()
        all_preds, all_true = [], []
        with torch.no_grad():
            for x, y in test_loader:
                all_preds.append(model(x.to(DEVICE)).argmax(1).cpu())
                all_true.append(y)
        all_preds = torch.cat(all_preds).numpy()
        all_true  = torch.cat(all_true).numpy()
        final_acc = accuracy_score(all_true, all_preds)

        mlflow.log_metric("final_test_acc", final_acc)
        mlflow.pytorch.log_model(model, artifact_path="model")

        print(f"  [{run_name}]  acc={final_acc:.4f}  params={n_params:,}")

    return {
        "subject":   subject,
        "window_size": window_size,
        "kernels":   kernels,
        "num_stages": num_stages,
        "history":   history,
        "preds":     all_preds,
        "true":      all_true,
        "label_map": label_map,
        "final_acc": final_acc,
    }

## Evaluation over multiple subjects

Jeder Proband bekommt ein eigenes subject-spezifisches Modell (Standard-Protokoll in der Literatur).
Ergebnisse werden gesammelt und am Ende verglichen.

In [ ]:
def get_finished_runs(experiment_name: str) -> set[str]:
    """Return the set of run_names that completed successfully in this experiment."""
    client = mlflow.tracking.MlflowClient()
    exp = client.get_experiment_by_name(experiment_name)
    if exp is None:
        return set()
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string="attributes.status = 'FINISHED'",
    )
    return {r.info.run_name for r in runs}


# Grid search — skips already-finished runs on resume
combos     = list(itertools.product(CONV_CONFIG, WINDOW_SIZES, NUM_MULTISCALE_STAGES))
total      = len(SUBJECTS) * len(combos)
finished   = get_finished_runs(MLFLOW_EXPERIMENT)
skipped    = sum(
    1 for s in SUBJECTS for kernels, window_size, num_stages in combos
    if f"db{DB}_s{s}_w{window_size}_k{'_'.join(map(str, kernels))}_stages{num_stages}" in finished
)
print(f"Total runs   : {total}")
print(f"Already done : {skipped}  (will be skipped)")
print(f"Remaining    : {total - skipped}")

results = []
run_idx = 0
for subject in SUBJECTS:
    for kernels, window_size, num_stages in combos:
        run_idx += 1
        run_name = f"db{DB}_s{subject}_w{window_size}_k{'_'.join(map(str, kernels))}_stages{num_stages}"

        if run_name in finished:
            print(f"[{run_idx}/{total}] SKIP (already finished): {run_name}")
            continue

        print(f"\n[{run_idx}/{total}] subject={subject}  kernels={kernels}  "
              f"window={window_size}  stages={num_stages}")
        r = train_and_evaluate(DB, subject, window_size, kernels, num_stages)
        results.append(r)

print("\nAll runs complete.")

In [ ]:
import pandas as pd

# Load ALL finished runs from MLflow (works correctly after a resume)
client = mlflow.tracking.MlflowClient()
exp    = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
all_runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
)

records = []
for r in all_runs:
    p = r.data.params
    records.append({
        "subject":     int(p.get("subject", -1)),
        "kernels":     p.get("kernels", ""),
        "window_size": int(p.get("window_size", -1)),
        "num_stages":  int(p.get("num_stages", -1)),
        "final_acc":   r.data.metrics.get("final_test_acc", float("nan")),
        "n_params":    int(p.get("n_params", 0)),
        "run_name":    r.info.run_name,
    })

df = pd.DataFrame(records).sort_values(["subject", "kernels", "window_size", "num_stages"])
print(f"Loaded {len(df)} finished runs\n")
print(df.to_string(index=False))

# Mean ± std per hyperparameter config across subjects
pivot = (df.groupby(["kernels", "window_size", "num_stages"])["final_acc"]
           .agg(["mean", "std", "count"])
           .reset_index()
           .sort_values("mean", ascending=False))
print("\nRanked by mean accuracy across subjects:")
print(pivot.to_string(index=False))

In [ ]:
# --- Bar chart: mean accuracy per config ---
fig, ax = plt.subplots(figsize=(14, 5))
labels_bar = [f"k={r['kernels']}\nw={r['window_size']} s={r['num_stages']}"
              for _, r in pivot.iterrows()]
ax.bar(range(len(pivot)), pivot["mean"], yerr=pivot["std"], capsize=4)
ax.set_xticks(range(len(pivot)))
ax.set_xticklabels(labels_bar, fontsize=8)
ax.set_ylabel("Mean Test Accuracy")
ax.set_title("Hyperparameter comparison (mean ± std over subjects)")
plt.tight_layout()
plt.show()

# --- MLflow UI hint ---
print("\nStart the MLflow UI to compare all runs interactively:")
print("  mlflow ui --port 5000")
print(f"  → http://127.0.0.1:5000  (experiment: '{MLFLOW_EXPERIMENT}')")